Build the master table (rossmann_master_train) cleanly by joining and feature engineering for forecasting.

1 Parse date fields, create time-based features

2 Join store metadata into sales data

3 Engineer promotion and competition features

4 Add holiday/event flags (basic version for now)

5 Create and save master table in rcg_store_catalog.rcg_store_schema

In [0]:
%sql
-- 1. Parse Date Fields and Basic Features
CREATE OR REPLACE TEMP VIEW train_with_features AS
SELECT 
    t.Store,
    t.Date,
    t.Sales,
    t.Customers,
    t.Open,
    t.Promo,
    t.StateHoliday,
    t.SchoolHoliday,
    
    -- Extract date parts
    CAST(t.Date AS DATE) AS Date_parsed,
    YEAR(CAST(t.Date AS DATE)) AS Year,
    MONTH(CAST(t.Date AS DATE)) AS Month,
    DAYOFWEEK(CAST(t.Date AS DATE)) AS DayOfWeek,
    WEEKOFYEAR(CAST(t.Date AS DATE)) AS WeekOfYear,
    
    -- Flags
    CASE WHEN DAYOFWEEK(CAST(t.Date AS DATE)) IN (1, 7) THEN 1 ELSE 0 END AS IsWeekend

FROM rcg_store_catalog.rcg_store_schema.rossman_store_sales_train AS t;


In [0]:
%sql
-- 2. Merge Store Metadata
CREATE OR REPLACE TEMP VIEW train_with_store_features AS
SELECT 
    tf.*,
    s.StoreType,
    s.Assortment,
    s.CompetitionDistance,
    s.CompetitionOpenSinceMonth,
    s.CompetitionOpenSinceYear,
    s.Promo2,
    s.Promo2SinceWeek,
    s.Promo2SinceYear
FROM train_with_features tf
LEFT JOIN rcg_store_catalog.rcg_store_schema.rossman_store s
ON tf.Store = s.Store;


In [0]:
%sql
-- 3. Engineer Competition Duration and Promo2 Activity
CREATE OR REPLACE TEMP VIEW train_feature_engineered AS
SELECT 
    *,
    
    -- Competition Open Duration in months
    CASE 
        WHEN CompetitionOpenSinceYear IS NOT NULL AND CompetitionOpenSinceMonth IS NOT NULL
        THEN DATEDIFF(Date_parsed, MAKE_DATE(CompetitionOpenSinceYear, CompetitionOpenSinceMonth, 1)) / 30
        ELSE NULL
    END AS MonthsSinceCompetitionOpen,
    
    -- Promo2 Active Flag
    CASE 
        WHEN Promo2 = 1 AND (
            (Promo2SinceYear < Year) OR (Promo2SinceYear = Year AND Promo2SinceWeek <= WeekOfYear)
        )
        THEN 1
        ELSE 0
    END AS IsPromo2Active

FROM train_with_store_features;


In [0]:
%sql
-- 4. Create Final Master Table
CREATE OR REPLACE TABLE rcg_store_catalog.rcg_store_schema.rossmann_store_sales_master_train AS
SELECT 
    Store,
    Date_parsed AS Date,
    Sales,
    Customers,
    Open,
    Promo,
    StateHoliday,
    SchoolHoliday,
    Year,
    Month,
    DayOfWeek,
    WeekOfYear,
    IsWeekend,
    
    StoreType,
    Assortment,
    CompetitionDistance,
    CompetitionOpenSinceMonth,
    CompetitionOpenSinceYear,
    MonthsSinceCompetitionOpen,
    Promo2,
    Promo2SinceWeek,
    Promo2SinceYear,
    IsPromo2Active

FROM train_feature_engineered;


In [0]:
%sql

select * from rcg_store_catalog.rcg_store_schema.rossman_store_sales_train limit 10

Step-by-Step SQL for Preparing Master Test Table

Step 1: Parse Dates and Basic Features

In [0]:
%sql
-- 1. Parse Date Fields and Basic Time Features
CREATE OR REPLACE TEMP VIEW test_with_features AS
SELECT 
    t.Id,
    t.Store,
    t.Date,
    t.Open,
    t.Promo,
    t.StateHoliday,
    t.SchoolHoliday,
    
    -- Extract date parts
    CAST(t.Date AS DATE) AS Date_parsed,
    YEAR(CAST(t.Date AS DATE)) AS Year,
    MONTH(CAST(t.Date AS DATE)) AS Month,
    DAYOFWEEK(CAST(t.Date AS DATE)) AS DayOfWeek,
    WEEKOFYEAR(CAST(t.Date AS DATE)) AS WeekOfYear,
    
    -- Flags
    CASE WHEN DAYOFWEEK(CAST(t.Date AS DATE)) IN (1, 7) THEN 1 ELSE 0 END AS IsWeekend

FROM rcg_store_catalog.rcg_store_schema.rossman_store_sales_test AS t;


In [0]:
%sql
select * from test_with_features

 Step 2: Join Store Metadata

In [0]:
%sql
-- 2. Merge Store Metadata into Test Data
CREATE OR REPLACE TEMP VIEW test_with_store_features AS
SELECT 
    tf.*,
    s.StoreType,
    s.Assortment,
    s.CompetitionDistance,
    s.CompetitionOpenSinceMonth,
    s.CompetitionOpenSinceYear,
    s.Promo2,
    s.Promo2SinceWeek,
    s.Promo2SinceYear
FROM test_with_features tf
LEFT JOIN rcg_store_catalog.rcg_store_schema.rossman_store s
ON tf.Store = s.Store;


Step 3: Feature Engineering for Competition and Promo2

In [0]:
%sql
-- 3. Engineer Competition Duration and Promo2 Active Flag
CREATE OR REPLACE TEMP VIEW test_feature_engineered AS
SELECT 
    *,
    
    -- Competition Open Duration in months
    CASE 
        WHEN CompetitionOpenSinceYear IS NOT NULL AND CompetitionOpenSinceMonth IS NOT NULL
        THEN DATEDIFF(Date_parsed, MAKE_DATE(CompetitionOpenSinceYear, CompetitionOpenSinceMonth, 1)) / 30
        ELSE NULL
    END AS MonthsSinceCompetitionOpen,
    
    -- Promo2 Active Flag
    CASE 
        WHEN Promo2 = 1 AND (
            (Promo2SinceYear < Year) OR (Promo2SinceYear = Year AND Promo2SinceWeek <= WeekOfYear)
        )
        THEN 1
        ELSE 0
    END AS IsPromo2Active

FROM test_with_store_features;


 Step 4: Save the Final Master Test Table

In [0]:
%sql
-- 4. Create Final Master Test Table
CREATE OR REPLACE TABLE rcg_store_catalog.rcg_store_schema.rossmann_store_sales_master_test AS
SELECT 
    Id,
    Store,
    Date_parsed AS Date,
    Open,
    Promo,
    StateHoliday,
    SchoolHoliday,
    Year,
    Month,
    DayOfWeek,
    WeekOfYear,
    IsWeekend,
    
    StoreType,
    Assortment,
    CompetitionDistance,
    CompetitionOpenSinceMonth,
    CompetitionOpenSinceYear,
    MonthsSinceCompetitionOpen,
    Promo2,
    Promo2SinceWeek,
    Promo2SinceYear,
    IsPromo2Active

FROM test_feature_engineered;


In [0]:
%sql

select * from rcg_store_catalog.rcg_store_schema.rossmann_store_sales_master_test